# Train the YOLO26 tennis-ball detector
Local replay uses pretrained YOLO26 for people/rackets/pose. This notebook fine-tunes a separate ball detector. Default: YOLO26s, 1280 px, 100 epochs, patience 20, batch 4 → 2 → 1 if GPU memory is insufficient. Training and datasets live on Colab's temporary disk; recoverable checkpoints go to Drive.

Public dataset: abdullahtarek/tennis_analysis, Roboflow tennis-ball-detection v6, CC BY 4.0. Keep attribution with derivatives.

## 1. Setup
Select **Runtime → Change runtime type → GPU**. This installs the versioned project wheel from the public GitHub release, including the same model and export code used locally.

In [ ]:
from google.colab import files, drive
from pathlib import Path
import subprocess, sys
WHEEL = 'https://github.com/Paulyang5049/tennis-ai-local/releases/download/v0.2.0/tennis_ai_local-0.2.0-py3-none-any.whl'
subprocess.run([sys.executable, '-m', 'pip', 'install', WHEEL], check=True)
print('If pip replaced an already imported torch/numpy, restart the session before continuing.')

In [ ]:
import torch, importlib.metadata
assert torch.cuda.is_available(), 'Choose a GPU runtime'
print('GPU:', torch.cuda.get_device_name(0))
print({n: importlib.metadata.version(n) for n in ['torch', 'ultralytics', 'numpy']})
drive.mount('/content/drive')
DRIVE = Path('/content/drive/MyDrive/TennisAI')
DRIVE.mkdir(parents=True, exist_ok=True)
print('Checkpoints will be saved to', DRIVE)

## 2. Download and audit
The repository's original 428/100/50 split is **not** blindly reused. Exact decoded-image duplicates are removed and conservative filename families are kept together. Filename families do not prove video identity: inspect `audit.json`, and use separate real phone/broadcast clips for final validation.

In [ ]:
from tennis_ai.training import prepare_ball, label_preview
DATA = Path('/content/tennis-ball')
try:
    data_yaml, audit = prepare_ball(DATA)
    print({k:v for k,v in audit.items() if k != 'manifest'})
except Exception as error:
    raise RuntimeError('Dataset unavailable or audit failed. The local COCO baseline remains usable; fix the data issue before training.') from error
label_preview(DATA)

## 3. Smoke test
Run one short epoch before committing to full training. This does not overwrite the recoverable full-training checkpoint.

In [ ]:
from tennis_ai.training import train_ball
train_ball(data_yaml, '/content/ball-runs', DRIVE/'ball', smoke=True)

## 4. Train or resume
Set `RESUME=True` after a disconnect to use Drive's last checkpoint. Re-run setup and data preparation in a fresh session first.

In [ ]:
import shutil
RESUME = False
checkpoint = None
if RESUME:
    checkpoint = Path('/content/ball-resume.pt')
    shutil.copy2(DRIVE/'ball'/'last.pt', checkpoint)
run, training_result = train_ball(data_yaml, '/content/ball-runs', DRIVE/'ball', resume=checkpoint)
print('Completed:', run)

## 5. Compare baseline and candidate
Both detectors are evaluated against the same tennis-ball labels, with the baseline's COCO sports-ball class mapped correctly. Validation selects a candidate; test results are reported separately. Image recall is not trajectory coverage.

In [ ]:
import json
from tennis_ai.training import evaluate_ball_images
best = DRIVE/'ball'/'best.pt'
report = {}
for split in ['val', 'test']:
    report[split] = {
        'baseline': evaluate_ball_images('yolo26s.pt', DATA, split, baseline=True),
        'candidate': evaluate_ball_images(best, DATA, split),
    }
a, b = report['val']['baseline'], report['val']['candidate']
report['image_candidate_pass'] = b['recall'] > a['recall'] and b['fp'] <= a['fp']
report['promotion'] = 'Candidate only: validate held-out video coverage and false detections before selecting locally.'
(DRIVE/'ball'/'evaluation.json').write_text(json.dumps(report, indent=2))
print(json.dumps(report, indent=2))

## 6. Export to the Mac
Download and unzip this bundle inside the local project's `models/` folder. In the app, enter its folder in **Custom Colab model bundles**, verify it, then analyze the same held-out clips with baseline and candidate. Keep the baseline unless observed ball coverage improves without more false detections.

In [ ]:
from tennis_ai.artifacts import package
bundle = package(best, DRIVE/'ball-bundle', 'ball', 1280, report)
archive = shutil.make_archive('/content/ball-bundle', 'zip', bundle)
files.download(archive)